# Entrenamiento de sistemas de IA mediante Aprendizaje por Refuerzo para tareas de razonamiento  


## Modelo parametrizado

Un **modelo** es una función
$f_\theta:X \to Y$
que recibe una entrada $x\in X$ y produce una salida $f_\theta(x)\in Y$.  
El subíndice $\theta$ indica que el comportamiento de la función depende de un conjunto de parámetros.

- $X$: espacio de entradas (por ejemplo, vectores numéricos, imágenes o secuencias de tokens).
- $Y$: espacio de salidas (por ejemplo, una clase, un número, una secuencia de tokens o una acción).
- $\theta$: parámetros ajustables (pesos y sesgos).

Idea clave: “Aprender” significa elegir $\theta$ para que $f_\theta$ haga lo que queremos según algún criterio medible.


## Datos y objetivo de entrenamiento

En muchos casos se dispone de ejemplos:
$D=\{(x_i,y_i)\}_{i=1}^n$
donde $x_i$ es la entrada y $y_i$ es la salida deseada (la “respuesta correcta” o “etiqueta”). Esto es el escenario típico del aprendizaje supervisado.

Para medir qué tan bien se comporta el modelo se define una pérdida (o error), por ejemplo $\ell(\hat{y}, y)$, donde $\hat{y} = f_{\theta}(x)$ es la predicción del modelo. La pérdida total suele ser un promedio:

$$
L(\theta) = \frac{1}{n}\sum_{i=1}^{n} \ell\big(f_{\theta}(x_i), y_i\big).
$$

El entrenamiento busca:

$$
\theta^{*} = \arg\min_{\theta} L(\theta).
$$


## Optimización: descenso por gradiente

Queremos minimizar una función de pérdida $L(\theta)$ (donde $\theta$ son los parámetros).  
La idea del descenso por gradiente sale de una aproximación de primer orden (Taylor) alrededor de $\theta$.

**Aproximación local (Taylor de primer orden)**
Para un pequeño cambio $\Delta\theta$, se tiene:

$$
L(\theta + \Delta\theta)
\approx
L(\theta) + \nabla_{\theta}L(\theta)^\top \Delta\theta.
$$

Esto dice: *cerca de $\theta$, la pérdida cambia casi linealmente*, y la dirección que más aumenta $L$ es el gradiente $\nabla_\theta L(\theta)$.

**Elegir una dirección que disminuya $L$**

Si elegimos $\Delta\theta$ en la dirección opuesta al gradiente:

$$
\Delta\theta = -\eta \,\nabla_{\theta}L(\theta), \qquad \eta>0,
$$

entonces sustituyendo en Taylor:

$$
L(\theta + \Delta\theta)
\approx
L(\theta) + \nabla_{\theta}L(\theta)^\top\big(-\eta \nabla_{\theta}L(\theta)\big)
=
L(\theta) - \eta \,\|\nabla_{\theta}L(\theta)\|^2.
$$

Como $\|\nabla_{\theta}L(\theta)\|^2 \ge 0$ y $\eta>0$, esto sugiere que (para $\eta$ suficientemente pequeña) **la pérdida disminuye**.

De aquí sale la actualización estándar:

$$
\theta \leftarrow \theta - \eta\,\nabla_\theta L(\theta).
$$

donde:

- $\nabla_\theta L(\theta)$ es el vector de derivadas parciales de $L$ respecto a cada parámetro.
- $\eta>0$ es la tasa de aprendizaje (tamaño del paso).

En redes profundas, $L(\theta)$ es una composición de muchas funciones.  
Para ver el mecanismo matemático, consideremos una composición simple:

$$
f_{\theta}(x) = g_{\theta_2}\big(h_{\theta_1}(x)\big),
\qquad
\hat{y}=f_{\theta}(x),
\qquad
L(\theta)=\ell(\hat{y},y).
$$

Define la variable intermedia:

$$
z = h_{\theta_1}(x),
\qquad
\hat{y} = g_{\theta_2}(z).
$$

**Gradiente respecto a $\theta_2$**

Por regla de la cadena:

$$
\nabla_{\theta_2} L
=
\frac{\partial L}{\partial \hat{y}}
\;\frac{\partial \hat{y}}{\partial \theta_2}
=
\Big(\nabla_{\hat{y}} \ell(\hat{y},y)\Big)\Big(\nabla_{\theta_2} g_{\theta_2}(z)\Big).
$$

**Gradiente respecto a $\theta_1$** 

Aquí $L$ depende de $\theta_1$ a través de $z=h_{\theta_1}(x)$, entonces:

$$
\nabla_{\theta_1} L
=
\frac{\partial L}{\partial z}\;\frac{\partial z}{\partial \theta_1}.
$$

Pero

$$
\frac{\partial L}{\partial z}
=
\frac{\partial L}{\partial \hat{y}}\;\frac{\partial \hat{y}}{\partial z}
=
\Big(\nabla_{\hat{y}} \ell(\hat{y},y)\Big)\Big(\nabla_{z} g_{\theta_2}(z)\Big).
$$

Por tanto:

$$
\nabla_{\theta_1} L
=
\Big(\nabla_{\hat{y}} \ell(\hat{y},y)\Big)\Big(\nabla_{z} g_{\theta_2}(z)\Big)\Big(\nabla_{\theta_1} h_{\theta_1}(x)\Big).
$$

cada bloque aporta su derivada, y se van multiplicando “hacia atrás” (de la salida hacia la entrada).  
Eso es exactamente backpropagation.


## Neurona (unidad básica) y por qué necesitamos no linealidad

Una **neurona** (o unidad) recibe una entrada $x \in \mathbb{R}^d$ y calcula primero una combinación afín (lineal + sesgo):

$$
z = w^\top x + b,
$$

y luego aplica una función de activación (típicamente no lineal) para producir la salida:

$$
y = \sigma(z),
$$

donde:

- $w \in \mathbb{R}^d$ son los **pesos**,
- $b \in \mathbb{R}$ es el **sesgo**,
- $\sigma:\mathbb{R}\to\mathbb{R}$ es la **activación**.

Un ejemplo muy usado es ReLU:

$$
\sigma(z)=\max\{0,z\}.
$$

**Por qué es necesaria la no linealidad:**  
Si $\sigma$ fuera la identidad (es decir, $\sigma(z)=z$), entonces cada capa sería solo una transformación afín. Al componer varias capas afines, el resultado sigue siendo una transformación afín; por ejemplo, sin sesgos para simplificar,

$$
x \mapsto W_2(W_1x) = (W_2W_1)x.
$$

Esto significa que, aunque apiles muchas capas, el modelo no gana “potencia” expresiva: seguiría comportándose como un modelo lineal. En cambio, al introducir una activación no lineal $\sigma$, la composición deja de ser lineal y la red puede aproximar relaciones mucho más complejas.


## Red neuronal multicapa (MLP) como composición de capas

Una red *feed-forward* (MLP) con $L$ capas puede escribirse como una composición iterada. Definimos la entrada como

$$
h_0 = x,
$$

y para cada capa oculta $\ell = 1, \dots, L-1$:

$$
h_\ell = \sigma\big(W_\ell h_{\ell-1} + b_\ell\big).
$$

Finalmente, una forma común de la **capa de salida** (por ejemplo, en regresión) es:

$$
f_\theta(x)= W_L h_{L-1} + b_L.
$$

En conjunto, los parámetros del modelo son:

$$
\theta = \{(W_\ell, b_\ell)\}_{\ell=1}^{L}.
$$



## Representación de texto: tokens y embeddings

Un modelo de lenguaje no “entiende” letras directamente. Para poder trabajar con el texto, primero lo convierte en números. El proceso típico tiene tres pasos: **tokenización**, **vocabulario** y **embeddings**.

**Tokenización (pasar texto a tokens)**

Dado un texto, se divide en piezas llamadas **tokens**. En la práctica, los tokens suelen ser **sub-palabras** (no necesariamente palabras completas).  
Después, a cada token se le asigna un **índice entero**.

Si el texto queda representado por la secuencia de índices

$$
(t_1,t_2,\dots,t_n),
$$

entonces $t_i$ es el índice del token en la posición $i$.

**Vocabulario**

El vocabulario es el conjunto de todos los tokens que el modelo puede reconocer. Lo denotamos por $V$, y su tamaño es $|V|$.  
Cada token del vocabulario tiene un índice en $\{1,2,\dots,|V|\}$ (a veces se usa $\{0,1,\dots,|V|-1\}$, depende de la implementación).

**Embeddings**

Un índice por sí solo no contiene “información” útil para el modelo. Por eso, cada token se representa mediante un vector en $\mathbb{R}^d$.  
Para esto se aprende una matriz de embeddings

$$
E \in \mathbb{R}^{|V|\times d}.
$$

La fila $t$ de esta matriz es el embedding del token con índice $t$. Es decir:

$$
e_t = E[t] \in \mathbb{R}^d.
$$

Entonces, para la secuencia $(t_1,\dots,t_n)$, se obtienen los vectores

$$
e_{t_1}, e_{t_2}, \dots, e_{t_n}.
$$

Si se apilan como filas, se forma la matriz de entrada

$$
X =
\begin{bmatrix}
e_{t_1}\\
e_{t_2}\\
\vdots\\
e_{t_n}
\end{bmatrix}
\in \mathbb{R}^{n\times d}.
$$

Aquí $n$ es el número de tokens del texto y $d$ es la dimensión de cada embedding.

## La necesidad de “contexto” en secuencias

En lenguaje, el significado de una palabra depende del contexto. Por ejemplo, “banco” puede significar institución financiera o asiento. Un modelo necesita producir representaciones donde cada token incorpore información de otros tokens relevantes. Esto se logra con **atención**.


## Transformer: arquitectura basada en auto-atención

Un **Transformer** procesa una secuencia representada por

$$
X \in \mathbb{R}^{n\times d},
$$

donde $n$ es el número de tokens y $d$ es la dimensión del embedding.  
El Transformer está formado por bloques que se repiten, y la operación principal dentro de cada bloque es la **auto-atención** (*self-attention*).


**Construcción de $Q$, $K$ y $V$**

A partir de $X$ se construyen tres matrices mediante proyecciones lineales:

$$
Q = XW_Q,\qquad K = XW_K,\qquad V = XW_V,
$$

con

- $W_Q, W_K \in \mathbb{R}^{d\times d_k}$,
- $W_V \in \mathbb{R}^{d\times d_v}$,

por lo tanto

$$
Q, K \in \mathbb{R}^{n\times d_k},\qquad V \in \mathbb{R}^{n\times d_v}.
$$

**Intuición**
- $Q$ (*queries*) describe lo que cada token “pregunta” o “busca”.
- $K$ (*keys*) describe con qué “etiqueta” se presenta cada token.
- $V$ (*values*) es la información que se va a combinar para producir la salida.


**Puntajes de atención y pesos**

Primero se calcula una matriz de puntajes comparando cada query con cada key:

$$
S = \frac{QK^\top}{\sqrt{d_k}} \in \mathbb{R}^{n\times n}.
$$

El factor $\sqrt{d_k}$ se usa para evitar que los puntajes crezcan demasiado cuando $d_k$ es grande, lo cual ayuda a que la función *softmax* no quede saturada.

Después, se transforman los puntajes en **pesos de atención** aplicando *softmax* por filas:

$$
A = \mathrm{softmax}(S),
$$

es decir, para cada $i$:

$$
A_{ij}=\frac{e^{S_{ij}}}{\sum_{m=1}^{n} e^{S_{im}}}.
$$

Cada fila de $A$ suma 1, y $A_{ij}$ indica cuánto peso le da la posición $i$ a la posición $j$.



**Mezcla de valores (salida de atención)**

La salida de auto-atención se obtiene mezclando los valores con esos pesos:

$$
\mathrm{Attn}(X) = AV.
$$

**Interpretación:** para cada posición $i$, el vector de salida es una combinación de los vectores $V_1,\dots,V_n$.  
Los coeficientes de esa combinación están en la fila $i$ de $A$, es decir, en $A_{i1},\dots,A_{in}$.

## Multi-head attention (varias atenciones en paralelo)

En lugar de calcular una sola auto-atención, un Transformer usa **$H$ cabezas** (*heads*) en paralelo. La idea es que cada cabeza aprende una forma distinta de comparar tokens.

Para cada cabeza $h \in \{1,\dots,H\}$ se usan matrices de proyección diferentes:

$$
Q^{(h)} = XW_Q^{(h)},\qquad
K^{(h)} = XW_K^{(h)},\qquad
V^{(h)} = XW_V^{(h)}.
$$

Luego, la salida de la cabeza $h$ se define como:

$$
\mathrm{head}^{(h)} \;=\;
\mathrm{softmax}\!\left(\frac{Q^{(h)}(K^{(h)})^\top}{\sqrt{d_k}}\right)V^{(h)}.
$$

Finalmente, se concatenan las salidas de todas las cabezas y se aplica una proyección lineal:

$$
\mathrm{MHA}(X)
=
\mathrm{Concat}\!\big(\mathrm{head}^{(1)},\dots,\mathrm{head}^{(H)}\big)\,W_O.
$$

**Por qué usar varias cabezas:**  
Cada cabeza puede enfocarse en un tipo de relación diferente entre tokens (por ejemplo, dependencias locales, referencias a tokens lejanos, patrones sintácticos o relaciones más lógicas). Al combinar varias, el modelo obtiene una representación más rica.



**Positional encoding (cómo se representa el orden)**

La auto-atención, por sí sola, no tiene forma de saber qué token está antes o después: si se permuta el orden de los tokens, el mecanismo de atención no “se da cuenta” automáticamente.  
Por eso se agrega información de posición.

Una forma estándar es sumar a cada embedding un vector de posición $p_i \in \mathbb{R}^d$:

$$
\widetilde{X}_i = X_i + p_i,\qquad i=1,\dots,n.
$$

En forma matricial, si $P \in \mathbb{R}^{n\times d}$ apila los vectores de posición como filas, entonces:

$$
\widetilde{X} = X + P.
$$

Los vectores $p_i$ pueden ser **sinusoidales** (fijos) o **aprendidos** (parámetros entrenables). Lo importante es que el modelo reciba suficiente información para distinguir el orden de la secuencia.

## Bloque Transformer (estructura típica)

Un bloque Transformer estándar combina dos subcapas:

1) **Multi-head attention (MHA)**  
2) **MLP** (una red feed-forward aplicada token por token)

En la práctica se usan **normalización** y **conexiones residuales**. Una forma común (llamada *pre-norm*) se escribe así.

Primero, la salida de la subcapa de atención:

$$
H = X + \mathrm{MHA}\big(\mathrm{LN}(X)\big),
$$

después, la salida de la subcapa MLP:

$$
Y = H + \mathrm{MLP}\big(\mathrm{LN}(H)\big).
$$

Aquí:

- $\mathrm{LN}$ es **LayerNorm** (normalización por características).
- Las conexiones residuales (los términos $X + \cdot$ y $H + \cdot$) ayudan a estabilizar el entrenamiento en redes profundas, porque permiten que la información y los gradientes fluyan mejor a través de muchas capas.
- La MLP se aplica de forma **independiente en cada posición** (misma transformación para todos los tokens, pero sin mezclar posiciones).

## Modelos de lenguaje autoregresivos (qué optimizan)

Un modelo de lenguaje **autoregresivo** aprende a predecir el siguiente token usando todos los tokens anteriores como contexto.

Si el texto (ya tokenizado) es

$$
(t_1, t_2, \dots, t_n),
$$

el modelo define, para cada posición $i$, una distribución de probabilidad del token $t_i$ condicionada a los tokens previos:

$$
p_\theta\!\left(t_i \mid t_{<i}\right),
\qquad
\text{donde } t_{<i} = (t_1,\dots,t_{i-1}).
$$

### Función objetivo (máxima verosimilitud)

Durante el entrenamiento se busca que el modelo asigne alta probabilidad al token correcto en cada paso. Esto se hace maximizando la probabilidad conjunta de la secuencia:

$$
p_\theta(t_1,\dots,t_n)
=
\prod_{i=1}^{n} p_\theta\!\left(t_i \mid t_{<i}\right).
$$

Maximizar esta cantidad es equivalente a minimizar la **pérdida de entropía cruzada** (negative log-likelihood):

$$
L(\theta) = -\sum_{i=1}^{n} \log p_\theta\!\left(t_i \mid t_{<i}\right).
$$

En palabras: en cada posición $i$ el modelo “propone” probabilidades para el siguiente token, y se penaliza cuando el token verdadero tiene probabilidad baja.

Este objetivo hace que el modelo aprenda regularidades del lenguaje y produzca texto coherente.  
Pero, estrictamente, lo que optimiza sigue siendo **predecir tokens**. En tareas de planeación o razonamiento secuencial, a veces no basta con predecir: lo que interesa es **tomar decisiones** para alcanzar una meta (y ahí entra el aprendizaje por refuerzo).

## De predecir texto a elegir acciones

Hay tareas donde el sistema debe producir una **secuencia de decisiones**, y la calidad de esa secuencia se evalúa hasta el final (o con señales muy parciales durante el proceso). Por ejemplo:

- resolver un acertijo en varios pasos,
- planear una ruta,
- construir un procedimiento que satisfaga restricciones,
- generar una cadena de razonamiento que lleve a una respuesta correcta.

En estos casos, la retroalimentación natural no es “cuál era el siguiente token”, sino una **señal numérica de desempeño**: éxito o fracaso, distancia a una meta, penalización por violar reglas, costo por tiempo, etc.

Por eso conviene cambiar el punto de vista: en lugar de pensar en “predicción del siguiente token”, se piensa en un **agente** que elige acciones paso a paso dentro de un **entorno**, y recibe recompensas.  
